# Phase 3: Verify Fine-Tuned Model

**Hardware:** GPU P100 (Settings → Accelerator → GPU P100).
**What this does:** Compares [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) of your fine-tuned model vs the baseline (`google/siglip-so400m-patch14-384`) on image-caption pairs from your dataset. The fine-tuned model should score higher.

**Prerequisite:** Run `02_train_siglip.ipynb` first.

**Before running:**
1. Set accelerator to GPU P100 (Settings → Accelerator → GPU P100).
2. Add the following secrets via Add-ons → Secrets — enable notebook access for each:
   - `HF_TOKEN` — HF [write-access token](https://huggingface.co/docs/hub/en/security-tokens#what-are-user-access-tokens).
   - `HF_USERNAME` — your HF username.
   - `FINETUNED_REPO` — repo ID of the fine-tuned model output from Notebook 2.
   - `CAPTIONED_DATASET` — repo ID of the captioned dataset output from Notebook 1.
   - If your dataset is gated, accept the access request on the HF Hub page first.
3. Run all cells top to bottom with **Shift+Enter**.

**Pass condition:** Fine-tuned avg similarity > baseline avg similarity.
**After passing:** Update your indexer's model name to the fine-tuned repo ID, delete cached index files, and restart the application.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets Pillow torch

In [ ]:
# Cell 2 — Authenticate and set config
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import torch

secrets = UserSecretsClient()

# Secrets — add all of these via Add-ons → Secrets in the Kaggle notebook UI
HF_TOKEN          = secrets.get_secret("HF_TOKEN")
HF_USERNAME       = secrets.get_secret("HF_USERNAME")
FINETUNED_REPO    = secrets.get_secret("FINETUNED_REPO")    # fine-tuned model output from Notebook 2
CAPTIONED_DATASET = secrets.get_secret("CAPTIONED_DATASET") # captioned dataset output from Notebook 1

BASELINE = "google/siglip-so400m-patch14-384"
device   = "cuda" if torch.cuda.is_available() else "cpu"

# Authenticate — required for gated datasets
login(token=HF_TOKEN)

print("HF_TOKEN:    ", HF_TOKEN[:8] + "..." if HF_TOKEN else "ERROR: not set")
print("HF_USERNAME: ", HF_USERNAME if HF_USERNAME else "ERROR: not set")
print("Fine-tuned:  ", FINETUNED_REPO)
print("Dataset:     ", CAPTIONED_DATASET)
print("Device:      ", device)

In [ ]:
# Cell 3 — Load both models
from transformers import AutoProcessor, AutoModel

def load_model(repo):
    m = AutoModel.from_pretrained(repo, token=HF_TOKEN).to(device).eval()
    p = AutoProcessor.from_pretrained(repo, token=HF_TOKEN)
    return m, p

print("Loading fine-tuned model...")
ft_model, ft_proc = load_model(FINETUNED_REPO)
print("Loading baseline model...")
bl_model, bl_proc = load_model(BASELINE)
print("Both models loaded")

In [ ]:
# Cell 4 — Load test pairs
from datasets import load_dataset

ds = load_dataset(CAPTIONED_DATASET, split="train[:10]", token=HF_TOKEN)
print(f"Loaded {len(ds)} test pairs")
print("Sample caption:", ds[0]["caption"])

In [ ]:
# Cell 5 — Compare similarity scores
import torch.nn.functional as F

def get_similarity(model, processor, image, text):
    inp = processor(images=image, text=[text], return_tensors="pt").to(device)
    with torch.no_grad():
        img_emb = F.normalize(model.get_image_features(
            pixel_values=inp["pixel_values"]), dim=-1)
        txt_emb = F.normalize(model.get_text_features(
            input_ids=inp["input_ids"],
            attention_mask=inp["attention_mask"]), dim=-1)
    return (img_emb * txt_emb).sum().item()

ft_sims, bl_sims = [], []
for i, row in enumerate(ds):
    img     = row["image"].convert("RGB")
    caption = row["caption"]
    ft_sim  = get_similarity(ft_model, ft_proc, img, caption)
    bl_sim  = get_similarity(bl_model, bl_proc, img, caption)
    ft_sims.append(ft_sim)
    bl_sims.append(bl_sim)
    print(f"[{i+1:2d}] FT={ft_sim:.4f}  BL={bl_sim:.4f}  diff={ft_sim-bl_sim:+.4f}")

print(f"\nFine-tuned avg similarity: {sum(ft_sims)/len(ft_sims):.4f}")
print(f"Baseline  avg similarity:  {sum(bl_sims)/len(bl_sims):.4f}")
if sum(ft_sims) > sum(bl_sims):
    print("\nPASS — fine-tuned model scores higher on your domain data.")
    print(f"\nNext step: update your indexer's model name to '{FINETUNED_REPO}'")
    print("Then delete cached index files and restart the application to reindex.")
else:
    print("\nREVIEW — fine-tuned model did not outperform baseline.")
    print("Check training logs: loss should have decreased below 0.8 by epoch 3.")